# Phase 06b: Data Discovery, Validation & Mathematical Feasibility Analysis

**Project:** APMC Mandi Market Intelligence System  
**Repository:** `ankit-choubey/Mandi_Prediction`  
**Author:** Priyanshu  
**Objective:** Establish, with empirical evidence from real Indian agricultural market data, whether the proposed expanded Mandi Prediction System is mathematically, statistically, and operationally feasible.

---

## 1. Setup & Environment Verification
Strictly configure single-threaded math libraries to prevent memory fragmentation on Windows systems.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
import gc
import time
from datetime import timedelta
import numpy as np
import pandas as pd
from scipy import stats
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    mean_absolute_error, root_mean_squared_error
)

# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.data_loader import load_and_clean_mandi_data

print("Environment verified successfully.")

## 2. Ingestion & Canonical Dataset Audit
Load raw APMC mandi records across 5 Uttar Pradesh districts using Strategy 2 (DEC-008) duplicate exclusion, preserving commercial `Grade`.

In [ ]:
t0 = time.time()
base_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
df = load_and_clean_mandi_data(data_dir=base_dir, min_series_records=20, exclude_ambiguous_same_grade=True)
print(f"Loaded {len(df):,} clean observations across {df['series_id'].nunique():,} series in {time.time()-t0:.2f}s.")
df.head()

## 3. Multi-Horizon Data Availability & Modal Price Stickiness ($T+1, T+3, T+7, T+14, T+30$)
Evaluate exact calendar matching continuity, boundary truncations, and the prevalence of 0.00% modal price stickiness.

In [ ]:
horizons = [1, 3, 7, 14, 30]
series_code, _ = pd.factorize(df['series_id'])
df['series_code'] = series_code.astype(np.int32)

lookup = df[['series_code', 'Arrival_Date', 'Modal_Price']].drop_duplicates().copy()
lookup['Modal_Price'] = lookup['Modal_Price'].astype(np.float32)

results = []
df_t7 = None

for h in horizons:
    lean_df = pd.DataFrame({
        'series_code': df['series_code'],
        'Arrival_Date': df['Arrival_Date'],
        'Modal_Price': df['Modal_Price'].astype(np.float32),
        'Min_Price': df['Min_Price'].astype(np.float32),
        'Max_Price': df['Max_Price'].astype(np.float32),
        'District': df['District'],
        'Commodity': df['Commodity'],
        'Market': df['Market']
    })
    lean_df['_target_date'] = lean_df['Arrival_Date'] + timedelta(days=h)
    
    merged = pd.merge(
        lean_df,
        lookup.rename(columns={'Modal_Price': f'Modal_Price_T{h}', 'Arrival_Date': '_target_date'}),
        on=['series_code', '_target_date'],
        how='left'
    )
    
    total_records = len(df)
    max_date = df['Arrival_Date'].max()
    boundary_truncated = int((df['Arrival_Date'] > (max_date - timedelta(days=h))).sum())
    eligible_origins = total_records - boundary_truncated
    matched_count = int(merged[f'Modal_Price_T{h}'].notna().sum())
    match_rate_eligible = (matched_count / eligible_origins) * 100.0 if eligible_origins > 0 else 0.0
    
    valid = merged.dropna(subset=[f'Modal_Price_T{h}']).copy()
    valid[f'Return_{h}'] = ((valid[f'Modal_Price_T{h}'] - valid['Modal_Price']) / valid['Modal_Price']) * 100.0
    
    sticky_pct = (valid[f'Return_{h}'].round(4) == 0.0).mean() * 100.0
    norm_fixed = ((valid[f'Return_{h}'] >= -2.5) & (valid[f'Return_{h}'] <= 2.5)).mean() * 100.0
    
    results.append({
        'Horizon': f'T+{h}',
        'Holding Days': h,
        'Eligible Match Rate (%)': round(match_rate_eligible, 2),
        'Sticky 0.00% Quotes (%)': round(sticky_pct, 2),
        'Normal Class [-2.5, +2.5] (%)': round(norm_fixed, 2),
        'Mean Return (%)': round(float(valid[f'Return_{h}'].mean()), 2),
        'Std Dev (%)': round(float(valid[f'Return_{h}'].std()), 2)
    })
    
    if h == 7:
        df_t7 = valid.copy()
        
    del lean_df, merged, valid
    gc.collect()

multi_h_df = pd.DataFrame(results)
multi_h_df

## 4. Classification Threshold Alternatives at $T+7$
Compare Fixed $\pm 2.5\%$, Wide $\pm 5\%$, Commodity Volatility, Quantile Terciles, and Economic Hurdle Rate.

In [ ]:
df_comp = df_t7[['Commodity', 'Return_7']].copy()

# 1. Fixed +/- 2.5%
t_fixed = np.where(df_comp['Return_7'] > 2.5, 2, np.where(df_comp['Return_7'] < -2.5, 0, 1))

# 2. Fixed +/- 5.0%
t_fixed_5 = np.where(df_comp['Return_7'] > 5.0, 2, np.where(df_comp['Return_7'] < -5.0, 0, 1))

# 3. Commodity Volatility
comm_stds = df_comp.groupby('Commodity')['Return_7'].std().fillna(df_comp['Return_7'].std())
comm_thresh = df_comp['Commodity'].map(comm_stds) * 0.5
t_comm_vol = np.where(df_comp['Return_7'] > comm_thresh, 2, np.where(df_comp['Return_7'] < -comm_thresh, 0, 1))

# 4. Quantile Terciles
def assign_terciles(s):
    q1, q2 = s.quantile(0.3333), s.quantile(0.6667)
    return np.where(s > q2, 2, np.where(s < q1, 0, 1))
t_terciles = df_comp.groupby('Commodity')['Return_7'].transform(assign_terciles)

# 5. Economic Hurdle Rate
t_econ = np.where(df_comp['Return_7'] > 3.5, 2, np.where(df_comp['Return_7'] < 0.0, 0, 1))

schemes = [
    ('Fixed +/-2.5% (DEC-015)', t_fixed),
    ('Fixed +/-5.0% (Wide)', t_fixed_5),
    ('Commodity Volatility (+/-0.5 sigma)', t_comm_vol),
    ('Quantile Terciles (33/33/33)', t_terciles),
    ('Economic Hurdle (+3.5% / 0.0%)', t_econ)
]

thresh_results = []
for name, arr in schemes:
    vc = pd.Series(arr).value_counts(normalize=True).to_dict()
    thresh_results.append({
        'Scheme': name,
        'Unfavorable (%)': round(vc.get(0, 0) * 100, 2),
        'Normal (%)': round(vc.get(1, 0) * 100, 2),
        'Favorable (%)': round(vc.get(2, 0) * 100, 2),
        'Majority Baseline (%)': round(max(vc.values()) * 100, 2),
        'Imbalance Ratio': round(max(vc.values()) / min(vc.values()), 2)
    })

pd.DataFrame(thresh_results)

## 5. Multi-Horizon Continuous Regression Benchmark vs Persistence
Benchmarking LightGBM Regressor against Zero-Return Persistence across all 5 horizons with Diebold-Mariano and Wilcoxon tests.

In [ ]:
reg_benchmark_csv = os.path.join(base_dir, 'backing', 'phase_06b', 'multi_horizon_regression_benchmark.csv')
if os.path.exists(reg_benchmark_csv):
    reg_df = pd.read_csv(reg_benchmark_csv)
    display(reg_df[['horizon', 'persistence_mae_pct', 'lightgbm_mae_pct', 'mae_delta_pct', 'lightgbm_beats_persistence', 'diebold_mariano_p_value', 'scientific_conclusion']])
else:
    print("Run backing/phase_06b/run_phase06b_experiments.py to generate regression benchmarks.")

## 6. Cross-District Spatial Generalization (Leave-One-District-Out)
Evaluating out-of-district spatial transferability across Ghaziabad, Jhansi, Kanpur, Lucknow, and Unnao.

In [ ]:
lodo_csv = os.path.join(base_dir, 'backing', 'phase_06b', 'cross_district_generalization.csv')
if os.path.exists(lodo_csv):
    lodo_df = pd.read_csv(lodo_csv)
    display(lodo_df[['held_out_district', 'test_majority_baseline_acc_pct', 'lodo_test_accuracy_pct', 'accuracy_gain_over_baseline_pct', 'mcnemar_p_value', 'generalization_verdict']])
else:
    print("Run backing/phase_06b/run_phase06b_experiments.py to generate LODO results.")

## 7. Crop Storage Economic Hurdle Rate Analysis
Evaluate whether holding produce for 7 days clears total storage, handling, shrinkage, and carrying cost friction across Perishables, Semi-perishables, and Grains.

In [ ]:
cat_csv = os.path.join(base_dir, 'backing', 'phase_06b', 'crop_category_hurdle_rate_analysis.csv')
if os.path.exists(cat_csv):
    cat_df = pd.read_csv(cat_csv)
    display(cat_df[['crop_category', 'mean_required_price_increase_pct', 'actual_mean_price_change_pct', 'empirical_storage_win_rate_pct', 'loss_frequency_pct', 'economic_viability_verdict']])
else:
    print("Run backing/phase_06b/run_phase06b_experiments.py to generate crop hurdle rate results.")

## 8. Final Conclusions & Verification Summary

1. **Continuous Price Regression Disqualified:** Persistence beats ML regression across 100% of horizons ($T+1$ to $T+30$).
2. **Threshold Imbalance Established:** Fixed $\pm 2.5\%$ traps 60%–80% of data in Normal. Quantile terciles should be adopted for pure ML benchmarking.
3. **Pan-India Myth Dispelled:** Cross-district transfer collapses in Bundelkhand (Jhansi), proving models must be trained by Agro-Climatic Zones.
4. **Tactical Storage Prohibited:** 7-day holding loses money in 87%–93% of decisions across grains and perishables. Operational farmer deployment remains strictly prohibited.
5. **Data Foundation Identified:** Immediate acquisition priority is CEDA Agmarknet for arrival volume backfilling.